# Implement a Query Validation Process

In this notebook, we will review how to set a validation process for SQL queries.

<figure>
 <img src="../assets/chapter_3_03.png" width="90%" align="center"/></a>
<figcaption> SQL Validation </figcaption>
</figure>

<br>
<br />

In [1]:
from typing import Iterable, Type

from sqlglot import parse, exp
from sqlglot.expressions import Expression


def is_query_valid(
    query: str,
    *,
    allowed_statements: Iterable[Type[Expression]],
    disallowed_statements: Iterable[Type[Expression]],
    dialect: str | None = None,
) -> bool:
    """
    Validate a SQL query using SQLGlot against allow / deny expression policies.

    Parameters
    ----------
    query : str
        SQL query to validate
    allowed_statements : Iterable[Type[Expression]]
        Expression types allowed at the top level
    disallowed_statements : Iterable[Type[Expression]]
        Expression types that are forbidden anywhere in the AST
    dialect : str, optional
        SQL dialect (e.g. "duckdb", "postgres", "snowflake")

    Returns
    -------
    bool
        True if the SQL query complies with the policy, False otherwise
    """
    try:
        statements = parse(query, dialect=dialect)
    except Exception:
        # Invalid SQL → reject
        return False

    allowed_statements = tuple(allowed_statements)
    disallowed_statements = tuple(disallowed_statements)

    for statement in statements:
        # 1. Enforce top-level allowlist
        if not isinstance(statement, allowed_statements):
            return False

        # 2. Reject forbidden expressions anywhere in the AST
        for node in statement.walk():
            if isinstance(node, disallowed_statements):
                return False

    return True


In [2]:
read_only_allowed  = (
    exp.Select,
    exp.With,
    exp.Except,
    exp.Show,
    exp.Describe,
)

read_only_disallowed = (
    exp.Insert,
    exp.Update,
    exp.Delete,
    exp.Create,
    exp.Drop,
    exp.Alter,
    exp.TruncateTable,
    exp.Merge,
    exp.Grant,
    exp.Revoke,
    exp.Analyze,
)


In [3]:
query = "SELECT * FROM air_traffic"

is_query_valid(
    query = query,
    allowed_statements=read_only_allowed,
    disallowed_statements=read_only_disallowed,
)

True

In [4]:
query = "DROP TABLE air_traffic"

is_query_valid(
    query = query,
    allowed_statements=read_only_allowed,
    disallowed_statements=read_only_disallowed,
)

False